# Paged Attention: Managing the KV Cache Like Virtual Memory

The full code of [Paged Attention: Managing the KV Cache Like Virtual Memory](https://residual-stream.lvtan71.workers.dev/posts/paged-attention-explained/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/paged-attention-explained) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/paged-attention-explained.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 4 min end to end on the RTX 3090 machine that saved the outputs below.
The last script, kernel.py, is a Metal kernel and runs only on an Apple-silicon Mac; the rest run anywhere.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 3 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q torch transformers huggingface_hub pandas pyarrow

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "paged-attention-explained"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/paged-attention-explained


## `waste.py`

How much of a KV cache holds real tokens under four ways of reserving memory, on ShareGPT and Alpaca request lengths. Writes `sharegpt_lengths.json` and `alpaca_lengths.json` (both included here).

In [2]:
!{sys.executable} waste.py

ShareGPT: 5000 requests, prompt 220 tokens on average, answer 204


  max    real tokens  17.6% of reserved slots;   2048 slots per running request,    7 running at once


  pow2   real tokens  47.7% of reserved slots;    757 slots per running request,   20 running at once


  oracle real tokens  63.0% of reserved slots;    573 slots per running request,   27 running at once


  paged  real tokens  98.0% of reserved slots;    369 slots per running request,   42 running at once


Alpaca: 5000 requests, prompt 18 tokens on average, answer 58


  max    real tokens   3.4% of reserved slots;   2048 slots per running request,    7 running at once


  pow2   real tokens  41.9% of reserved slots;    168 slots per running request,   93 running at once


  oracle real tokens  57.0% of reserved slots;    124 slots per running request,  127 running at once


  paged  real tokens  90.4% of reserved slots;     78 slots per running request,  201 running at once


The post printed:

```text
ShareGPT: 5000 requests, prompt 220 tokens on average, answer 204
  max    real tokens  17.6% of reserved slots;   2048 slots per running request,    7 running at once
  pow2   real tokens  47.7% of reserved slots;    757 slots per running request,   20 running at once
  oracle real tokens  63.0% of reserved slots;    573 slots per running request,   27 running at once
  paged  real tokens  98.0% of reserved slots;    369 slots per running request,   42 running at once
Alpaca: 5000 requests, prompt 18 tokens on average, answer 58
  max    real tokens   3.4% of reserved slots;   2048 slots per running request,    7 running at once
  pow2   real tokens  41.9% of reserved slots;    168 slots per running request,   93 running at once
  oracle real tokens  57.0% of reserved slots;    124 slots per running request,  127 running at once
  paged  real tokens  90.4% of reserved slots;     78 slots per running request,  201 running at once
```

## `same.py`

Two prompts decoded through the paged cache, checked token for token against the contiguous cache.

In [3]:
!{sys.executable} same.py

one block: 16 tokens x 28 layers x K and V x 2 heads x 128 dimensions x 2 bytes = 458,752 bytes


prompt 0: 103 tokens in blocks [0, 3, 5, 7, 9, 11, 13]; same 100 tokens as the contiguous cache: True


prompt 1: 110 tokens in blocks [1, 2, 4, 6, 8, 10, 12]; same 100 tokens as the contiguous cache: True
block size 16; 50 of 64 blocks free
' the ability to request a certain amount of data from a database, and then receive it in a batch. This is a way to improve the performance of a database system. In the context of a database'


The post printed:

```text
one block: 16 tokens x 28 layers x K and V x 2 heads x 128 dimensions x 2 bytes = 458,752 bytes
prompt 0: 103 tokens in blocks [0, 3, 5, 7, 9, 11, 13]; same 100 tokens as the contiguous cache: True
prompt 1: 110 tokens in blocks [1, 2, 4, 6, 8, 10, 12]; same 100 tokens as the contiguous cache: True
block size 16; 50 of 64 blocks free
' the ability to request a certain amount of data from a database, and then receive it in a batch. This is a way to improve the performance of a database system. In the context of a database'
```

## `share.py`

Four samples of one prompt sharing its blocks.

In [4]:
!{sys.executable} share.py

prompt: 44 tokens in 3 blocks, the last one 12 of 16 full


4 samples of 64 tokens: 22 blocks in use, against 28 without sharing (21% fewer); 3 copy-on-write copies
tables:
  [0, 1, 3, 6, 10, 14, 18]
  [0, 1, 4, 7, 11, 15, 19]
  [0, 1, 5, 8, 12, 16, 20]
  [0, 1, 2, 9, 13, 17, 21]


each sample the same as without sharing: [True, True, True, True]
  'A KV (Key-Value) cache needs its own memory manager because KV caches are typically small '
  "A KV (key-value) cache needs its own memory manager because the cache's memory usage and m"
  "A KV (Key-Value) cache needs its own memory manager to efficiently manage the cache's memo"
  'A Key-Value (KV) cache typically needs its own memory manager because it often operates at'


The post printed:

```text
prompt: 44 tokens in 3 blocks, the last one 12 of 16 full
4 samples of 64 tokens: 22 blocks in use, against 28 without sharing (21% fewer); 3 copy-on-write copies
tables:
  [0, 1, 3, 6, 10, 14, 18]
  [0, 1, 4, 7, 11, 15, 19]
  [0, 1, 5, 8, 12, 16, 20]
  [0, 1, 2, 9, 13, 17, 21]
each sample the same as without sharing: [True, True, True, True]
  'A KV (Key-Value) cache needs its own memory manager because KV caches are typically small '
  "A KV (key-value) cache needs its own memory manager because the cache's memory usage and m"
  "A KV (Key-Value) cache needs its own memory manager to efficiently manage the cache's memo"
  'A Key-Value (KV) cache typically needs its own memory manager because it often operates at'
```

## `prefix.py`

Prefix caching by chained block hashes.

In [5]:
!{sys.executable} prefix.py

A: 96 tokens, 6 full blocks; tokens shared with A from the start:  96; blocks found in the cache: 0, prompt tokens not recomputed: 0
B: 96 tokens, 6 full blocks; tokens shared with A from the start:  85; blocks found in the cache: 5, prompt tokens not recomputed: 80
C: 110 tokens, 6 full blocks; tokens shared with A from the start:   3; blocks found in the cache: 0, prompt tokens not recomputed: 0


The post printed:

```text
A: 96 tokens, 6 full blocks; tokens shared with A from the start:  96; blocks found in the cache: 0, prompt tokens not recomputed: 0
B: 96 tokens, 6 full blocks; tokens shared with A from the start:  85; blocks found in the cache: 5, prompt tokens not recomputed: 80
C: 110 tokens, 6 full blocks; tokens shared with A from the start:   3; blocks found in the cache: 0, prompt tokens not recomputed: 0
```

## `blocksize.py`

Waste and table length against block size. Reads the two length files.

In [6]:
!{sys.executable} blocksize.py

block   real tokens / reserved     blocks per request     B's 85 shared tokens
size    ShareGPT     Alpaca        ShareGPT   Alpaca      reused from the cache


   1     100.0%     100.0%         424.0     75.5       85


   4      99.6%      97.9%         106.4     19.2       84


   8      99.0%      95.3%          53.4      9.9       80


  16      98.0%      90.4%          27.0      5.2       80


  32      95.9%      82.4%          13.7      2.8       64


  64      92.0%      68.7%           7.1      1.7       64


 128      84.9%      48.8%           3.8      1.1        0


 256      73.0%      27.2%           2.2      1.0        0


The post printed:

```text
block   real tokens / reserved     blocks per request     B's 85 shared tokens
size    ShareGPT     Alpaca        ShareGPT   Alpaca      reused from the cache
   1     100.0%     100.0%         424.0     75.5       85
   4      99.6%      97.9%         106.4     19.2       84
   8      99.0%      95.3%          53.4      9.9       80
  16      98.0%      90.4%          27.0      5.2       80
  32      95.9%      82.4%          13.7      2.8       64
  64      92.0%      68.7%           7.1      1.7       64
 128      84.9%      48.8%           3.8      1.1        0
 256      73.0%      27.2%           2.2      1.0        0
```

## `kernel.py`

Decode attention through a block table, as a Metal kernel, timed against a contiguous cache.

**Note:** Apple-silicon Mac only (Metal).

In [7]:
!{sys.executable} kernel.py

Traceback (most recent call last):
  File "residual-stream-code/posts/paged-attention-explained/kernel.py", line 97, in <module>
    part_m = torch.empty(PARTS, device="mps")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
NotImplementedError: Could not run 'aten::empty.memory_format' with arguments from the 'MPS' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'aten::empty.memory_format' is only available for these backends: [CPU, CUDA, Meta, QuantizedCPU, QuantizedCUDA, QuantizedMeta, MkldnnCPU, SparseCPU, SparseCUDA, SparseMeta, SparseCsrCPU, SparseCsrCUDA, SparseCsrMeta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradHIP, AutogradXLA, A

The post printed:

```text
16 sequences x 4096 cached tokens, 64 MiB of keys and values per step
contiguous, no table: attention 1.16 ms (58 GB/s read), max error vs PyTorch 5.2e-08; copying alone 1.60 ms (84 GB/s read and written)
        attention, time vs contiguous     copying alone                max error
block   in order   scattered               in order   scattered        vs PyTorch
    1      1.00x      1.02x                  1.00x      1.01x           5.2e-08
    2      0.99x      1.00x                  1.00x      1.02x           5.2e-08
    4      1.00x      1.00x                  1.00x      1.01x           5.2e-08
    8      1.01x      1.01x                  1.00x      1.00x           5.2e-08
   16      1.00x      0.99x                  0.99x      1.00x           5.2e-08
   32      1.00x      1.00x                  0.99x      1.00x           5.2e-08
   64      1.00x      0.99x                  0.99x      1.00x           5.2e-08
  128      1.00x      0.98x                  1.00x      0.99x           5.2e-08
  256      1.01x      0.99x                  1.00x      1.00x           5.2e-08
```